**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Mise en place `polygoneMeteoFrance.shp`

# Création du Fichier du Polygone Météo (`polygoneMeteoFrance.shp`)

## 1. Objectif

Ce document décrit la méthodologie pour générer le fichier `polygoneMeteoFrance.shp`, qui représente l'enveloppe géographique globale de la zone d'étude. Ce polygone unique est utilisé par MAELIA pour y associer les données climatiques.

## 2. Méthodologie

Le processus consiste à :
1.  **Fusionner** l'ensemble des polygones du `parcellaire_enrichi.shp` en une seule entité géographique.
2.  **Calculer** les coordonnées du centroïde de cette nouvelle entité pour obtenir `POSX` et `POSY`.
3.  **Créer** les attributs requis (`ID_PDG`, `ALTI_MOY`) avec des valeurs fixes.

## 3. Fichiers en Entrée et en Sortie

* **Fichier en Entrée :** `data/sols/shapefiles/processed/parcellaire_enrichi.shp`
* **Fichier en Sortie :** `includes_sassemeV1/modeleCommun/meteo/polygoneMeteoFrance.shp`

## 4. Description des Attributs

| Variable | Description | Origine / Traitement |
| :--- | :--- | :--- |
| `ID_PDG` | Identifiant du polygone. | Valeur fixe **`'0001'`**, formatée sur quatre chiffres. |
| `POSX` | Coordonnée X du centroïde. | Calculée après la fusion de toutes les parcelles. |
| `POSY` | Coordonnée Y du centroïde. | Calculée après la fusion de toutes les parcelles. |
| `ALTI_MOY`| Altitude moyenne. | Valeur fixe `0.0`. |

In [1]:
import geopandas as gpd
from pathlib import Path

In [2]:
# --- CHEMINS ET IMPORTS ---
base_dir = Path.cwd().parent.resolve()

# Fichier en entrée
input_parcellaire_path = base_dir / "data" / "sols" / "shapefiles" / "processed" / "parcellaire_enrichi.shp"

# Fichier en sortie
output_meteo_poly_path = base_dir / "includes_sassemeV1" / "modeleCommun" / "meteo" / "polygoneMeteoFrance.shp"

In [3]:
# --- CHARGEMENT DES DONNÉES ---
try:
    gdf_parcellaire = gpd.read_file(input_parcellaire_path)
    print(f"✅ Parcellaire chargé avec {len(gdf_parcellaire)} polygones.")
except Exception as e:
    print(f"🚨 ERREUR lors du chargement du fichier : {e}")

✅ Parcellaire chargé avec 749 polygones.


In [5]:
# --- TRAITEMENT GÉOSPATIAL ---
# Fusionner tous les polygones en une seule géométrie
zone_etude_poly = gdf_parcellaire.unary_union
print("-> Toutes les parcelles ont été fusionnées en un seul polygone.")

# Calculer le centroïde de ce polygone unifié
centroid = zone_etude_poly.centroid
print(f"-> Centroïde de la zone d'étude calculé : ({centroid.x:.4f}, {centroid.y:.4f})")

C:\Users\Cheikhou\AppData\Local\Temp\ipykernel_16600\3189292394.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  zone_etude_poly = gdf_parcellaire.unary_union


-> Toutes les parcelles ont été fusionnées en un seul polygone.
-> Centroïde de la zone d'étude calculé : (336039.9758, 1603304.4780)


In [6]:
# --- CRÉATION DU GEODATAFRAME FINAL ---
# On crée un nouveau GeoDataFrame avec une seule ligne
gdf_meteo = gpd.GeoDataFrame(
    {
        'ID_PDG': ['0001'],
        'POSX': [centroid.x],
        'POSY': [centroid.y],
        'ALTI_MOY': [0.0]
    },
    geometry=[zone_etude_poly],
    crs=gdf_parcellaire.crs  # On conserve le CRS d'origine
)
print("✅ GeoDataFrame final pour la météo créé.")

✅ GeoDataFrame final pour la météo créé.


In [8]:
# --- SAUVEGARDE ---
output_meteo_poly_path.parent.mkdir(parents=True, exist_ok=True)
gdf_meteo.to_file(output_meteo_poly_path, driver='ESRI Shapefile', encoding='utf-8')
print(f"\n✅ Fichier 'polygoneMeteoFrance.shp' sauvegardé dans :\n   {output_meteo_poly_path}")


✅ Fichier 'polygoneMeteoFrance.shp' sauvegardé dans :
   C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\includes_sassemeV1\modeleCommun\meteo\polygoneMeteoFrance.shp


In [9]:
print("\nAperçu du fichier final :")
display(gdf_meteo)


Aperçu du fichier final :


,ID_PDG,POSX,POSY,ALTI_MOY,geometry
0,0001,336039.975838,1.603304e+06,0.0,"MULTIPOLYGON (((333627.355 1599982.626, 333629..."
